In [1]:
!pip install groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 642.0 kB/s eta 0:00:00a 0:00:01


In [3]:
import pandas as pd
from groq import Groq
import time

# 1. API Configuration
# BITTE DEINEN GROQ KEY HIER EINSETZEN
client = Groq(api_key="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")

# Current stable model on Groq in April 2026
MODEL_ID = "llama-3.3-70b-versatile" 

# 2. Load the Dataset
file_name = 'dataset_clean.csv'

try:
    df = pd.read_csv(file_name)
    total_questions = len(df)
    print(f"--- STARTING HIGH-SPEED GENERATION WITH {MODEL_ID} ---")
    
    answers = []
    
    for index, row in df.iterrows():
        question_id = row['id']
        prompt_text = row['prompt']
        
        # Expert instruction
        prompt_instruction = (
            f"Du bist ein Experte für österreichisches Steuerrecht. "
            f"Beantworte die folgende Frage kurz und präzise auf Deutsch. "
            f"Nenne, wenn möglich, die gesetzlichen Grundlagen (z.B. EStG, KStG, BAO): {prompt_text}"
        )
        
        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {"role": "system", "content": "Du bist ein präziser Steuerrechtsexperte für Österreich."},
                    {"role": "user", "content": prompt_instruction}
                ],
                model=MODEL_ID,
            )
            
            answer = chat_completion.choices[0].message.content.strip()
            answers.append(answer)
            print(f"Done: {question_id} ({index + 1}/{total_questions})")
            
        except Exception as e:
            # Handle rate limits or model errors
            error_str = str(e).lower()
            if "rate_limit" in error_str or "429" in error_str:
                print(f"⚠️ Rate Limit! Waiting 10 seconds...")
                time.sleep(10)
                # Simple retry
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": prompt_instruction}],
                    model=MODEL_ID,
                )
                answers.append(chat_completion.choices[0].message.content.strip())
            elif "404" in error_str or "decommissioned" in error_str:
                print("❌ Model ID issue. Trying fallback model 'llama-3.1-70b-versatile'...")
                MODEL_ID = "llama-3.1-70b-versatile" # Automatic switch
                continue # Re-run this loop iteration with new model
            else:
                print(f"!!! Error at {question_id}: {e}")
                answers.append("Error: Generation failed.")
        
        # Groq is very fast, but let's keep a tiny buffer
        time.sleep(0.4)

    # 3. Save results
    if len(answers) > 0:
        df_result = df.iloc[:len(answers)].copy()
        df_result['answer'] = answers
        output_file = 'model_1_inference_groq.csv'
        df_result[['id', 'answer']].to_csv(output_file, index=False)
        print(f"\n--- SUCCESS! Results saved to {output_file} ---")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

--- STARTING HIGH-SPEED GENERATION WITH llama-3.3-70b-versatile ---
Done: CORP-TAX-001 (1/643)
Done: CORP-TAX-002 (2/643)
Done: CORP-TAX-003 (3/643)
Done: CORP-TAX-004 (4/643)
Done: CORP-TAX-005 (5/643)
Done: CORP-TAX-006 (6/643)
Done: CORP-TAX-007 (7/643)
Done: CORP-TAX-008 (8/643)
Done: CORP-TAX-009 (9/643)
Done: CORP-TAX-010 (10/643)
Done: CORP-TAX-011 (11/643)
Done: CORP-TAX-012 (12/643)
Done: CORP-TAX-013 (13/643)
Done: CORP-TAX-014 (14/643)
Done: CORP-TAX-015 (15/643)
Done: CORP-TAX-016 (16/643)
Done: CORP-TAX-017 (17/643)
Done: CORP-TAX-018 (18/643)
Done: CORP-TAX-019 (19/643)
Done: CORP-TAX-020 (20/643)
Done: CORP-TAX-021 (21/643)
Done: CORP-TAX-022 (22/643)
Done: CORP-TAX-023 (23/643)
Done: CORP-TAX-024 (24/643)
Done: CORP-TAX-025 (25/643)
Done: CORP-TAX-026 (26/643)
Done: CORP-TAX-027 (27/643)
Done: CORP-TAX-028 (28/643)
Done: CORP-TAX-029 (29/643)
Done: CORP-TAX-030 (30/643)
Done: CORP-TAX-031 (31/643)
Done: CORP-TAX-032 (32/643)
Done: CORP-TAX-033 (33/643)
Done: CORP-TAX-03

In [5]:
import pandas as pd
from groq import Groq
import time
import os

# 1. API Configuration
client = Groq(api_key="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")
MODEL_ID = "llama-3.1-8b-instant" 

output_file = 'model_1_inference_groq.csv'
input_file = 'dataset_clean.csv'

try:
    df = pd.read_csv(input_file)
    
    # Fortschritt prüfen: Wo haben wir aufgehört?
    if os.path.exists(output_file):
        df_progress = pd.read_csv(output_file)
        start_index = len(df_progress)
        answers = df_progress['answer'].tolist()
        print(f"--- RESUMING FROM ROW {start_index} ---")
    else:
        start_index = 0
        answers = []
        print(f"--- STARTING FRESH WITH {MODEL_ID} ---")

    for index in range(start_index, len(df)):
        row = df.iloc[index]
        question_id = row['id']
        
        prompt_instruction = (
            f"Du bist ein Experte für österreichisches Steuerrecht. "
            f"Beantworte die Frage kurz auf Deutsch mit Gesetzesangaben: {row['prompt']}"
        )
        
        success = False
        while not success:
            try:
                chat_completion = client.chat.completions.create(
                    messages=[{"role": "user", "content": prompt_instruction}],
                    model=MODEL_ID,
                )
                answer = chat_completion.choices[0].message.content.strip()
                answers.append(answer)
                
                # Sofort speichern, damit bei einem Crash nichts verloren geht
                temp_df = df.iloc[:len(answers)].copy()
                temp_df['answer'] = answers
                temp_df[['id', 'answer']].to_csv(output_file, index=False)
                
                print(f"Done: {question_id} ({index + 1}/{len(df)})")
                success = True
                
            except Exception as e:
                if "429" in str(e):
                    print(f"⚠️ Limit erreicht bei {question_id}. Pause für 30 Sek...")
                    time.sleep(30)
                else:
                    print(f"!!! Kritischer Fehler bei {question_id}: {e}")
                    answers.append("Error: Generation failed.")
                    success = True # Überspringen, um nicht in Endlosschleife zu landen
        
        # Kleine Pause, um die "Requests per Minute" niedrig zu halten
        time.sleep(1.5)

    print(f"\n--- SUCCESS! Alle {len(df)} Fragen bearbeitet. ---")

except Exception as e:
    print(f"Ein unerwarteter Fehler ist aufgetreten: {e}")

--- STARTING FRESH WITH llama-3.1-8b-instant ---
Done: CORP-TAX-001 (1/643)
Done: CORP-TAX-002 (2/643)
Done: CORP-TAX-003 (3/643)
Done: CORP-TAX-004 (4/643)
Done: CORP-TAX-005 (5/643)
Done: CORP-TAX-006 (6/643)
Done: CORP-TAX-007 (7/643)
Done: CORP-TAX-008 (8/643)
Done: CORP-TAX-009 (9/643)
Done: CORP-TAX-010 (10/643)
Done: CORP-TAX-011 (11/643)
Done: CORP-TAX-012 (12/643)
Done: CORP-TAX-013 (13/643)
Done: CORP-TAX-014 (14/643)
Done: CORP-TAX-015 (15/643)
Done: CORP-TAX-016 (16/643)
Done: CORP-TAX-017 (17/643)
Done: CORP-TAX-018 (18/643)
Done: CORP-TAX-019 (19/643)
Done: CORP-TAX-020 (20/643)
Done: CORP-TAX-021 (21/643)
Done: CORP-TAX-022 (22/643)
Done: CORP-TAX-023 (23/643)
Done: CORP-TAX-024 (24/643)
Done: CORP-TAX-025 (25/643)
Done: CORP-TAX-026 (26/643)
Done: CORP-TAX-027 (27/643)
Done: CORP-TAX-028 (28/643)
Done: CORP-TAX-029 (29/643)
Done: CORP-TAX-030 (30/643)
Done: CORP-TAX-031 (31/643)
Done: CORP-TAX-032 (32/643)
Done: CORP-TAX-033 (33/643)
Done: CORP-TAX-034 (34/643)
Done: CO